
# 🏦 Stock Trading Notebook — Yahoo Finance (Ex.: **LULU**)

**Persona:** Programador especializado em fundamentos de mercados, focado em **otimizar ganhos** com **técnicas elaboradas de trading**.  
**Fonte de dados:** **Yahoo Finance** via `yfinance` (grátis).  
> Educacional — não é recomendação de investimento.



## 0) Instalação
Se estiver em um ambiente novo, rode:
```bash
pip install --quiet yfinance pandas matplotlib pytz
# Opcional (candles bonitos)
pip install --quiet mplfinance
# Opcional (indicadores prontos; o notebook tem fallback)
pip install --quiet pandas_ta
```


In [ ]:

# 1) Imports, config e utilidades
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta, timezone
import pytz
import matplotlib.pyplot as plt

# Tenta pandas_ta (opcional)
try:
    import pandas_ta as ta
    HAS_PTA = True
except Exception:
    HAS_PTA = False

TICKER = "LULU"        # exemplo: Lululemon Athletica Inc.
LOCAL_TZ = "America/New_York"  # mercado EUA

def to_local_tz(df, tz=LOCAL_TZ):
    if df is None or df.empty: 
        return df
    # yfinance já traz timezone; garantimos conversão
    if df.index.tz is None:
        df = df.tz_localize("UTC")
    return df.tz_convert(tz)

def yf_download(ticker, period, interval):
    df = yf.download(ticker, period=period, interval=interval, auto_adjust=True, progress=False)
    if df is None or df.empty:
        raise RuntimeError(f"Sem dados para {ticker} period={period} interval={interval}")
    df = df.rename(columns={"Open":"Open","High":"High","Low":"Low","Close":"Close","Volume":"Volume"})
    df.index.name = "Datetime"
    return to_local_tz(df)

def resample_ohlc(df, rule):
    rule = str(rule).lower()
    o = df["Open"].resample(rule).first()
    h = df["High"].resample(rule).max()
    l = df["Low"].resample(rule).min()
    c = df["Close"].resample(rule).last()
    v = df["Volume"].resample(rule).sum()
    out = pd.concat([o,h,l,c,v], axis=1).dropna()
    out.columns = ["Open","High","Low","Close","Volume"]
    return out

# Indicadores com fallback
def ema(series, span): return series.ewm(span=span, adjust=False).mean()

def macd_series(close, fast=12, slow=26, sig=9):
    if HAS_PTA:
        m = ta.macd(close, fast=fast, slow=slow, signal=sig).dropna()
        return m[f"MACD_{fast}_{slow}_{sig}"], m[f"MACDs_{fast}_{slow}_{sig}"], m[f"MACDh_{fast}_{slow}_{sig}"]
    macd_line = ema(close, fast) - ema(close, slow)
    signal = ema(macd_line, sig)
    hist = macd_line - signal
    return macd_line.dropna(), signal.dropna(), hist.dropna()

def rsi_series(close, length=14):
    if HAS_PTA:
        return ta.rsi(close, length=length).dropna()
    delta = close.diff()
    gain = (delta.clip(lower=0)).ewm(alpha=1/length, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1/length, adjust=False).mean()
    rs = gain / loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi.dropna()

def macd_signals(close):
    macd_line, signal, hist = macd_series(close)
    cross = np.sign(macd_line - signal).diff()
    last_cross_idx = cross.dropna().ne(0).iloc[::-1].idxmax() if not cross.dropna().empty else None
    last_signal = None
    if last_cross_idx is not None and last_cross_idx in cross.index:
        last_signal = "BUY" if cross.loc[last_cross_idx] > 0 else "SELL"
    bias = "BULLISH" if macd_line.iloc[-1] > signal.iloc[-1] else "BEARISH"
    return macd_line, signal, hist, last_signal, bias

def rsi_state(close):
    rsi = rsi_series(close)
    st = "NEUTRAL"
    if rsi.iloc[-1] >= 70: st = "OVERBOUGHT"
    elif rsi.iloc[-1] <= 30: st = "OVERSOLD"
    return rsi, st


In [ ]:

# 2) Coleta de dados por **período**
# Períodos solicitados: 3 meses, 1 mês, 7 dias, 1 dia e 4 horas

# a) 3 meses — intervalos de 1h funcionam bem para ações
df_3mo_1h = yf_download(TICKER, period="3mo", interval="1h")

# b) 1 mês — 30 minutos
df_1mo_30m = yf_download(TICKER, period="1mo", interval="30m")

# c) 7 dias — 15 minutos
df_7d_15m = yf_download(TICKER, period="7d", interval="15m")

# d) 1 dia — 5 minutos
df_1d_5m = yf_download(TICKER, period="1d", interval="5m")

# e) 4 horas — pegue 1d/5m e recorte as últimas 4h
df_4h_window = df_1d_5m.loc[df_1d_5m.index.max() - pd.Timedelta(hours=4):]

print("Amostras: 3mo/1h=", len(df_3mo_1h), "| 1mo/30m=", len(df_1mo_30m),
      "| 7d/15m=", len(df_7d_15m), "| 1d/5m=", len(df_1d_5m), "| 4h slice=", len(df_4h_window))


In [ ]:

# 3) Helpers de plot (candle com fallback) + função de EMA21 entry/exit
def plot_candles(df, title):
    try:
        import mplfinance as mpf
        mpf.plot(df, type="candle", volume=True, title=title, style="classic")
    except Exception:
        ax = df["Close"].plot(title=title, figsize=(10,4)); ax.set_xlabel("Tempo"); ax.set_ylabel("Preço"); plt.show()

def plot_lines(series_dict, title):
    plt.figure()
    for label, s in series_dict.items():
        s.dropna().plot(label=label)
    plt.title(title); plt.legend(); plt.xlabel("Tempo"); plt.ylabel("Valor"); plt.show()

def plot_ema21_signals(df, title):
    d = df.copy()
    d["EMA21"] = d["Close"].ewm(span=21, adjust=False).mean()
    cross_up   = (d["Close"] > d["EMA21"]) & (d["Close"].shift(1) <= d["EMA21"].shift(1))
    cross_down = (d["Close"] < d["EMA21"]) & (d["Close"].shift(1) >= d["EMA21"].shift(1))
    try:
        import mplfinance as mpf
        fig, ax = mpf.plot(d, type="candle", volume=False, returnfig=True, title=title, style="classic")
        ax.plot(d.index, d["EMA21"])
        ax.scatter(d.index[cross_up], d["Close"][cross_up])
        ax.scatter(d.index[cross_down], d["Close"][cross_down])
        fig.show()
    except Exception:
        ax = d[["Close","EMA21"]].plot(title=title, figsize=(10,4))
        ax.scatter(d.index[cross_up], d["Close"][cross_up])
        ax.scatter(d.index[cross_down], d["Close"][cross_down])
        plt.show()
    if cross_up.iloc[-1]:   print("Último evento: ENTRY (bullish cross).")
    elif cross_down.iloc[-1]: print("Último evento: EXIT (bearish cross).")
    else: print("Último evento: Sem cruzamento na última vela.")


## 3 meses (1h)

In [ ]:
plot_candles(df_3mo_1h, '3 meses (1h) — Candles')

In [ ]:

m,s,h,signal,bias = macd_signals(df_3mo_1h["Close"])
plot_lines({"MACD": m, "Signal": s, "Hist": h}, "3 meses (1h) — MACD")
print("MACD — Sinal:", signal, "| Tendência:", bias)


In [ ]:

r, st = rsi_state(df_3mo_1h["Close"])
plot_lines({"RSI": r}, "3 meses (1h) — RSI")
print("RSI — Estado:", st)


In [ ]:
plot_ema21_signals(df_3mo_1h, '3 meses (1h) — EMA21 Entry/Exit')

## 1 mês (30m)

In [ ]:
plot_candles(df_1mo_30m, '1 mês (30m) — Candles')

In [ ]:

m,s,h,signal,bias = macd_signals(df_1mo_30m["Close"])
plot_lines({"MACD": m, "Signal": s, "Hist": h}, "1 mês (30m) — MACD")
print("MACD — Sinal:", signal, "| Tendência:", bias)


In [ ]:

r, st = rsi_state(df_1mo_30m["Close"])
plot_lines({"RSI": r}, "1 mês (30m) — RSI")
print("RSI — Estado:", st)


In [ ]:
plot_ema21_signals(df_1mo_30m, '1 mês (30m) — EMA21 Entry/Exit')

## 7 dias (15m)

In [ ]:
plot_candles(df_7d_15m, '7 dias (15m) — Candles')

In [ ]:

m,s,h,signal,bias = macd_signals(df_7d_15m["Close"])
plot_lines({"MACD": m, "Signal": s, "Hist": h}, "7 dias (15m) — MACD")
print("MACD — Sinal:", signal, "| Tendência:", bias)


In [ ]:

r, st = rsi_state(df_7d_15m["Close"])
plot_lines({"RSI": r}, "7 dias (15m) — RSI")
print("RSI — Estado:", st)


In [ ]:
plot_ema21_signals(df_7d_15m, '7 dias (15m) — EMA21 Entry/Exit')

## 1 dia (5m)

In [ ]:
plot_candles(df_1d_5m, '1 dia (5m) — Candles')

In [ ]:

m,s,h,signal,bias = macd_signals(df_1d_5m["Close"])
plot_lines({"MACD": m, "Signal": s, "Hist": h}, "1 dia (5m) — MACD")
print("MACD — Sinal:", signal, "| Tendência:", bias)


In [ ]:

r, st = rsi_state(df_1d_5m["Close"])
plot_lines({"RSI": r}, "1 dia (5m) — RSI")
print("RSI — Estado:", st)


In [ ]:
plot_ema21_signals(df_1d_5m, '1 dia (5m) — EMA21 Entry/Exit')

## Janela 4 horas (5m)

In [ ]:
plot_candles(df_4h_window, 'Janela 4 horas (5m) — Candles')

In [ ]:

m,s,h,signal,bias = macd_signals(df_4h_window["Close"])
plot_lines({"MACD": m, "Signal": s, "Hist": h}, "Janela 4 horas (5m) — MACD")
print("MACD — Sinal:", signal, "| Tendência:", bias)


In [ ]:

r, st = rsi_state(df_4h_window["Close"])
plot_lines({"RSI": r}, "Janela 4 horas (5m) — RSI")
print("RSI — Estado:", st)


In [ ]:
plot_ema21_signals(df_4h_window, 'Janela 4 horas (5m) — EMA21 Entry/Exit')